In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


# feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURE_ENGINEERING.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *

### Load Model

In [4]:
model = joblib.load('Models/xgbModel.pkl')
features = joblib.load('Models/top_features.pkl')

### Load Data

In [5]:
pd.set_option('display.max_columns', None)
eplison = 0.000001

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s25['EXPECTED_USAGE_MIN'] = s25['USG_PCT_ROLLING_AVG_5'] * (s25['MIN_ROLLING_AVG_5'] + eplison)

s24 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
s24['EXPECTED_USAGE_MIN'] = s24['USG_PCT_ROLLING_AVG_5'] * (s24['MIN_ROLLING_AVG_5'] + eplison)
df = pd.concat([s25, s24]).sort_values(by='GAME_DATE')


date = '2025-03-11'
df = df[df['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
dfsData = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_points')]
df.tail()

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_14676/4012596075.py:15: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


Unnamed: 0  Unnamed: 0.2      PLAYER_NAME  PLAYER_ID      MATCHUP  \
26070       26070       20644.0       Seth Curry     203552    CHA @ MIA   
26069       26069       20648.0   Nick Smith Jr.    1641733    CHA @ MIA   
19868       19868       20673.0     Blake Wesley    1631104  SAS vs. DAL   
19867       19867       20670.0  Bismack Biyombo     202687  SAS vs. DAL   
633           633       20546.0     Terance Mann    1629611  ATL vs. PHI   

      TEAM_ABBREVIATION     TEAM_ID OPP_ABBREVIATION  HOME_GAME   GAME_ID  \
26070               CHA  1610612766              MIA          0  22400931   
26069               CHA  1610612766              MIA          0  22400931   
19868               SAS  1610612759              DAL          1  22400937   
19867               SAS  1610612759              DAL          1  22400937   
633                 ATL  1610612737              PHI          1  22400928   

        GAME_DATE WL  PTS  AST  REB  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  \
26070  2025-03-10  W    0    1    1    0    3   0.000     0     3      0.0   
26069  2025-03-10  W    4    1    1    2    6   0.333     0     2      0.0   
19868  2025-03-10  L    3    0    0    1    2   0.500     1     1      1.0   
19867  2025-03-10  L    3    1    5    1    1   1.000     0     0      NaN   
633    2025-03-10  W   19    4    6    9   13   0.692     0     2      0.0   

       FTM  FTA  FT_PCT  OREB  DREB  STL  BLK  TOV  PLUS_MINUS  FANTASY_PTS  \
26070    0    0     NaN     0     1    0    0    1         -12          1.7   
26069    0    0     NaN     0     1    0    0    1         -19          5.7   
19868    0    0     NaN     0     0    0    0    0          -7          3.0   
19867    1    2     0.5     0     5    0    1    0           1         13.5   
633      1    1     1.0     1     5    0    1    1          11         34.2   

       POINT_PER_SHOT       EFG START_POSITION  COMMENT  E_OFF_RATING  \
26070           0.000  0.000000            NaN      NaN          69.3   
26069           0.667  0.333333            NaN      NaN          69.5   
19868           1.500  0.750000            NaN      NaN          94.3   
19867           1.596  1.000000              C      NaN         129.9   
633             1.414  0.692308            NaN      NaN         130.2   

       E_DEF_RATING  NET_RATING  OREB_PCT  DREB_PCT  REB_PCT  AST_PCT  \
26070         102.7       -41.4      0.00     0.067    0.030    0.125   
26069         128.8       -65.6      0.00     0.056    0.026    0.200   
19868         122.3       -22.7      0.00     0.000    0.000    0.000   
19867         123.5         2.9      0.00     0.250    0.156    0.059   
633           114.9        14.9      0.03     0.147    0.090    0.133   

       EFG_PCT  AST_TOV  USG_PCT  TS_PCT  E_PACE    PACE    PIE  POSS  \
26070    0.000      1.0    0.125   0.000  100.02   96.62 -0.080    29   
26069    0.333      1.0    0.194   0.333   95.75   96.53  0.000    33   
19868    0.750      0.0    0.095   0.750   94.57   91.78  0.029    20   
19867    1.000      0.0    0.056   0.798  100.80  100.40  0.094    35   
633      0.692      4.0    0.163   0.707  110.42  108.23  0.129    74   

       PACE_PER40  E_USG_PCT    MIN   SPD  DIST  ORBC  DRBC  RBC  TCHS  SAST  \
26070       80.52      0.134  14.40  4.35  1.13     0     3    3    19     0   
26069       80.44      0.214  15.92  4.50  1.27     1     5    5    36     1   
19868       76.48      0.094  10.98  4.66  0.91     1     2    3    12     0   
19867       83.67      0.056  16.73  4.51  1.32     1     7    8    26     1   
633         90.19      0.166  32.82  4.39  2.57     3     6    9    48     1   

       FTAST  PASS  CFGM  CFGA  CFG_PCT  UFGM  UFGA  UFG_PCT  DFGM  DFGA  \
26070      0    15     0     0    0.000     0     3    0.000     1     1   
26069      0    29     1     3    0.333     1     3    0.333     0     1   
19868      0     9     0     1    0.000     1     1    1.000     0     0   
19867      0    24     1     1    1.000  

In [6]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singleBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleBookies

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
124606,Draymond Green,points,under,fanduel,10.5,-122,10.5,102,2025-03-11
124607,Draymond Green,points,under,draftkings,10.5,-110,10.5,102,2025-03-11
124608,Draymond Green,points,under,espnbet,10.5,-120,10.5,102,2025-03-11
124610,Draymond Green,points,under,betrivers,10.5,-113,10.5,102,2025-03-11
124616,Draymond Green,points,over,fanduel,10.5,-104,10.5,-102,2025-03-11
...,...,...,...,...,...,...,...,...,...
127136,Karlo Matković,points,over,espnbet,6.5,-105,6.5,106,2025-03-11
127137,Karlo Matković,points,over,draftkings,6.5,-105,6.5,106,2025-03-11
127144,Karlo Matković,points,under,betmgm,6.5,-125,6.5,-106,2025-03-11
127145,Karlo Matković,points,under,espnbet,6.5,-125,6.5,-106,2025-03-11


### Top EVs for single bets

In [7]:
results = single_bet(
    data=df,
    bookmakers=singleBookies,
    model=model,
    features=features,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
560,Taurean Prince,betmgm,points,17.5,100,under,6.003530,1,0.016,0.984,0.500,96.76,0.97,0.48,0.24,"(0.5, 16.5)"
637,Thomas Bryant,betmgm,points,11.5,-110,under,5.895394,1,0.069,0.931,0.524,77.74,0.86,0.43,0.21,"(0.7, 13.2)"
374,Javonte Green,betmgm,points,0.5,-120,over,1.891558,0,0.956,0.044,0.545,75.25,0.90,0.45,0.23,"(0.3, 18.2)"
741,Amir Coffey,draftkings,points,7.5,-110,over,10.458364,0,0.916,0.084,0.524,74.82,0.82,0.41,0.21,"(6.2, 14.6)"
739,Amir Coffey,espnbet,points,7.5,-110,over,10.458364,0,0.915,0.085,0.524,74.72,0.82,0.41,0.21,"(6.3, 14.6)"
742,Amir Coffey,fanduel,points,7.5,-110,over,10.458364,0,0.912,0.088,0.524,74.20,0.82,0.41,0.20,"(6.3, 14.6)"
409,Jalen Duren,betmgm,points,17.5,-115,under,10.001787,1,0.072,0.928,0.535,73.55,0.85,0.42,0.21,"(1.6, 19.9)"
270,Cameron Johnson,espnbet,points,12.5,115,over,16.579876,0,0.806,0.194,0.465,73.40,0.64,0.32,0.16,"(7.5, 25.7)"
116,Josh Hart,betmgm,points,21.5,-130,under,13.092574,1,0.020,0.980,0.565,73.31,0.95,0.48,0.24,"(5.1, 21.2)"
353,Trendon Watford,espnbet,points,4.5,105,over,8.336388,0,0.835,0.165,0.488,71.15,0.68,0.34,0.17,"(1.2, 16.8)"


### Top EVs for 2 leg bets

In [8]:
results = prizepickspairsEV(
    data=df,
    bookmakers=dfsData,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.60,OVER,0.882,0.118,"(2.0, 15.7)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",OVER/UNDER,0,0.8749,1.625,0.812
1,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",Max Christie,player_points,prizepicks,-137,16.5,under,13.35,UNDER,0.129,0.871,"(7.8, 18.7)",UNDER/UNDER,0,0.8647,1.594,0.797
2,Deni Avdija,player_points,prizepicks,-137,15.5,under,20.12,OVER,0.867,0.133,"(11.9, 28.3)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",OVER/UNDER,1,0.8601,1.580,0.790
3,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",Julian Champagnie,player_points,prizepicks,-137,7.5,over,12.71,OVER,0.867,0.133,"(3.6, 22.0)",UNDER/OVER,1,0.8597,1.579,0.790
4,Ja Morant,player_points,prizepicks,-137,27.0,over,20.85,UNDER,0.141,0.859,"(9.6, 32.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",UNDER/UNDER,1,0.8525,1.557,0.779
5,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",Tari Eason,player_points,prizepicks,-137,13.5,over,10.00,UNDER,0.154,0.846,"(3.4, 16.8)",UNDER/UNDER,0,0.8398,1.519,0.760
6,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.72,UNDER,0.156,0.844,"(0.6, 10.1)",UNDER/UNDER,0,0.8378,1.513,0.757
7,AJ Green,player_points,prizepicks,-137,6.5,under,3.54,UNDER,0.180,0.820,"(0.3, 9.5)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",UNDER/UNDER,0,0.8137,1.441,0.721
8,Harrison Barnes,player_points,prizepicks,-137,11.0,over,18.24,OVER,0.818,0.182,"(3.3, 34.6)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",OVER/UNDER,1,0.8111,1.433,0.717
9,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.08,UNDER,0.008,0.992,"(1.1, 19.5)",Obi Toppin,player_points,prizepicks,-137,7.5,under,11.60,OVER,0.817,0.183,"(3.0, 20.5)",UNDER/OVER,0,0.8105,1.432,0.716


In [5]:
threeLeg = prizepicks3LegEV(
    data=df,
    bookmakers=dfsData,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,SIDE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
105502,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",OVER/UNDER/UNDER,0,0.8030,3.818,0.764
351546,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Max Christie,player_points,prizepicks,-137,16.5,under,13.39,UNDER,0.133,0.867,"(7.9, 18.9)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/UNDER/UNDER,0,0.7726,3.636,0.727
105487,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Max Christie,player_points,prizepicks,-137,16.5,under,13.39,UNDER,0.133,0.867,"(7.9, 18.9)",OVER/UNDER/UNDER,0,0.7697,3.618,0.724
352051,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Obi Toppin,player_points,prizepicks,-137,7.5,under,12.55,OVER,0.862,0.138,"(3.7, 21.7)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/OVER/UNDER,1,0.7686,3.612,0.722
105500,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Obi Toppin,player_points,prizepicks,-137,7.5,under,12.55,OVER,0.862,0.138,"(3.7, 21.7)",OVER/UNDER/OVER,0,0.7657,3.594,0.719
352117,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",Patrick Williams,player_points,prizepicks,-137,8.5,over,4.05,UNDER,0.147,0.853,"(0.3, 11.8)",UNDER/UNDER/UNDER,0,0.7608,3.565,0.713
350812,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.49,UNDER,0.147,0.853,"(0.5, 9.9)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/UNDER/UNDER,0,0.7602,3.561,0.712
105505,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Patrick Williams,player_points,prizepicks,-137,8.5,over,4.05,UNDER,0.147,0.853,"(0.3, 11.8)",OVER/UNDER/UNDER,0,0.7579,3.547,0.709
105473,Ben Sheppard,player_points,prizepicks,-137,4.5,over,8.95,OVER,0.901,0.099,"(2.3, 16.0)",Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,7.5,under,4.49,UNDER,0.147,0.853,"(0.5, 9.9)",OVER/UNDER/UNDER,0,0.7574,3.544,0.709
350566,Jalen Williams,player_points,prizepicks,-137,21.5,over,9.85,UNDER,0.014,0.986,"(1.4, 20.2)",Julian Champagnie,player_points,prizepicks,-137,7.5,over,12.30,OVER,0.844,0.156,"(3.4, 21.6)",Paolo Banchero,player_points,prizepicks,-137,24.5,over,15.62,UNDER,0.096,0.904,"(3.3, 29.1)",UNDER/OVER/UNDER,1,0.7526,3.515,0.703
